# Tweet Sentiment Analysis with LSTM Models

This notebook compares three LSTM architectures for sentiment classification:
- LSTM
- Bidirectional LSTM
- Encoder-Decoder LSTM

**Dataset**: CardiffNLP Tweet Sentiment Evaluation (3-class: negative, neutral, positive)

## 1. Import Required Libraries

In [81]:
import os
import shutil

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.tensorflow

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    RepeatVector,
    TextVectorization,
)
from tensorflow.keras.models import Sequential

import shap
from lime.lime_text import LimeTextExplainer

import matplotlib.pyplot as plt
import seaborn as sns

## 2. Configuration and Reproducibility

In [82]:
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

CONFIG = {
    "max_words": 10000,
    "seq_len": 50,
    "embedding_dim": 64,
    "lstm_units": 64,
    "dropout_rate": 0.3,
    "epochs": 10,
    "batch_size": 64,
    "validation_split": 0.1,
    "test_size": 0.2,
    "optimizer": "adam",
    "loss_function": "sparse_categorical_crossentropy",
    "random_state": 42,
    "early_stopping_patience": 3
}

mlflow.set_experiment("Tweet_Sentiment_LSTM_Models")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "grid.color": "#eeeeee"
})

## 3. Load and Explore Data

In [83]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few examples:")
display(df.head())

Dataset shape: (45615, 2)

First few examples:


,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2


## 4. Data Preprocessing

In [84]:
X = df["text"].astype(str).values
y = df["label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training set label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

Training set size: 36492
Test set size: 9123
Training set label distribution:
0     5674
1    16539
2    14279
Name: count, dtype: int64


## 5. Text Vectorization

In [85]:
vectorizer = TextVectorization(
    max_tokens=CONFIG["max_words"],
    output_sequence_length=CONFIG["seq_len"]
)

vectorizer.adapt(X_train)

## 6. Define Training Function

In [86]:
def train_and_evaluate(model, name):
    """Trains, evaluates, and logs a TensorFlow/Keras model via MLflow.

    This function automatically handles model compilation, applies early stopping
    based on global configurations, trains the model on the available dataset, 
    calculates test set metrics, and syncs parameters, step-by-step histories, 
    and the final model artifact cleanly to an active MLflow tracking session.

    Args:
        model (tf.keras.Model): An uncompiled or compiled TensorFlow/Keras sequence
            or sequential model layout.
        name (str): Unique descriptive text identifier assigned to this specific 
            training run within the MLflow interface.

    Returns:
        tuple: A three-element tuple containing:
            - y_pred (numpy.ndarray): Class index predictions calculated from the
              test dataset array, shape (num_samples,).
            - acc (float): Overall test classification accuracy metric computed via 
              scikit-learn's accuracy_score.
            - history (tf.keras.callbacks.History): Full training history containing
              epoch-by-epoch tracking records of loss and training metrics.

    Raises:
        NameError: If underlying text arrays (X_train, y_train, X_test, y_test) 
            or the central dict `CONFIG` are missing from the global execution scope.
    """
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model_name": name,
            "max_words": CONFIG["max_words"],
            "seq_len": CONFIG["seq_len"],
            "embedding_dim": CONFIG["embedding_dim"],
            "lstm_units": CONFIG["lstm_units"],
            "dropout_rate": CONFIG["dropout_rate"],
            "epochs": CONFIG["epochs"],
            "batch_size": CONFIG["batch_size"],
            "optimizer": CONFIG["optimizer"],
            "loss_function": CONFIG["loss_function"],
            "early_stopping_patience": CONFIG["early_stopping_patience"],
        })

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=CONFIG["early_stopping_patience"],
            restore_best_weights=True
        )

        model.compile(
            optimizer=CONFIG["optimizer"],
            loss=CONFIG["loss_function"],
            metrics=["accuracy"]
        )

        history = model.fit(
            X_train,
            y_train,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            validation_split=CONFIG["validation_split"],
            callbacks=[early_stop],
            verbose=1
        )

        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        y_pred = np.argmax(model.predict(X_test), axis=1)
        acc = accuracy_score(y_test, y_pred)

        mlflow.log_metrics({
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
            "accuracy_score": acc,
        })

        for epoch, value in enumerate(history.history["loss"]):
            mlflow.log_metric("train_loss", value, step=epoch)
        for epoch, value in enumerate(history.history["accuracy"]):
            mlflow.log_metric("train_accuracy", value, step=epoch)
        for epoch, value in enumerate(history.history["val_loss"]):
            mlflow.log_metric("val_loss", value, step=epoch)
        for epoch, value in enumerate(history.history["val_accuracy"]):
            mlflow.log_metric("val_accuracy", value, step=epoch)

        mlflow.tensorflow.log_model(
            model=model,
            name=f"{name}_model"
        )

        print(f"\n{'='*50}")
        print(f"Model: {name}")
        print(f"Test Accuracy: {acc:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        print(f"{'='*50}")

        return y_pred, acc, history

## 7. Define Model Architectures

In [87]:
# Standard LSTM Model
lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    LSTM(CONFIG["lstm_units"]),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("LSTM Model Architecture:")
lstm_model.summary()

LSTM Model Architecture:


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_3            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [88]:
# Bidirectional LSTM Model
bilstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    Bidirectional(LSTM(CONFIG["lstm_units"])),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("\nBidirectional LSTM Model Architecture:")
bilstm_model.summary()


Bidirectional LSTM Model Architecture:


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_3            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_10 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [89]:
# Encoder-Decoder LSTM Model
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    
    # Encoder
    LSTM(CONFIG["lstm_units"]),
    
    # Decoder
    RepeatVector(CONFIG["seq_len"]),
    LSTM(CONFIG["lstm_units"], return_sequences=True),
    
    # Classification
    GlobalAveragePooling1D(),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("\nEncoder-Decoder LSTM Model Architecture:")
ed_lstm_model.summary()


Encoder-Decoder LSTM Model Architecture:


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_3            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_11 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_3 (RepeatVector)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 8. Train and Evaluate Models

In [90]:
results = {}
all_predictions = {}
all_histories = {} 

# 1. Standard LSTM
lstm_pred, lstm_acc, lstm_hist = train_and_evaluate(lstm_model, "LSTM")
results["LSTM"] = lstm_acc
all_predictions["LSTM"] = lstm_pred
all_histories["LSTM"] = lstm_hist

# 2. Bidirectional LSTM
bilstm_pred, bilstm_acc, bilstm_hist = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["Bi-LSTM"] = bilstm_acc
all_predictions["Bi-LSTM"] = bilstm_pred
all_histories["Bi-LSTM"] = bilstm_hist

# 3. Encoder-Decoder LSTM
ed_lstm_pred, ed_lstm_acc, ed_lstm_hist = train_and_evaluate(ed_lstm_model, "ED-LSTM")
results["ED-LSTM"] = ed_lstm_acc
all_predictions["ED-LSTM"] = ed_lstm_pred
all_histories["ED-LSTM"] = ed_lstm_hist

print("\nAll models trained and interpretability artifacts logged to MLflow successfully.")

Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4481 - loss: 1.0195 - val_accuracy: 0.4507 - val_loss: 1.0173
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4807 - loss: 0.9514 - val_accuracy: 0.5877 - val_loss: 0.8539
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6434 - loss: 0.7820 - val_accuracy: 0.5789 - val_loss: 0.8614
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7165 - loss: 0.6788 - val_accuracy: 0.6164 - val_loss: 0.9217
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7671 - loss: 0.5988 - val_accuracy: 0.6005 - val_loss: 0.9229
286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


2026/06/04 09:39:14 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: LSTM
Test Accuracy: 0.5804
Test Loss: 0.8613
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5903 - loss: 0.8576 - val_accuracy: 0.6342 - val_loss: 0.7846
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7092 - loss: 0.6616 - val_accuracy: 0.6279 - val_loss: 0.8028
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7555 - loss: 0.5716 - val_accuracy: 0.6205 - val_loss: 0.9086
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7935 - loss: 0.4923 - val_accuracy: 0.6079 - val_loss: 1.0932
286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


2026/06/04 09:39:38 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: Bi-LSTM
Test Accuracy: 0.6370
Test Loss: 0.7839
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.4461 - loss: 1.0194 - val_accuracy: 0.4507 - val_loss: 1.0169
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4503 - loss: 1.0093 - val_accuracy: 0.4619 - val_loss: 0.9660
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4698 - loss: 0.9277 - val_accuracy: 0.5211 - val_loss: 0.8985
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6001 - loss: 0.8226 - val_accuracy: 0.5710 - val_loss: 0.8567
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.7042 - loss: 0.6922 - val_accuracy: 0.6115 - val_loss: 0.8866
Epoch 6/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.7570 - loss: 0.6167 - val_accuracy: 0.6153 - val_loss: 0.9632
Epoch 7/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.7955 - loss: 0.5495 - val_accuracy: 0.6121 - val_loss: 1.0485
286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


2026/06/04 09:40:36 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: ED-LSTM
Test Accuracy: 0.5785
Test Loss: 0.8704

All models trained and interpretability artifacts logged to MLflow successfully.


## 9. Compare Results

In [91]:
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])

print("\n" + "="*50)
print("Overall Model Comparison")
print("="*50)
print(results_df.to_string(index=False))


Overall Model Comparison
  Model  Accuracy
   LSTM  0.580401
Bi-LSTM  0.636962
ED-LSTM  0.578538


## 10. Detailed Model Evaluation

In [92]:
label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    ))

    # Confusion matrix analysis
    cm = confusion_matrix(y_test, y_pred)
    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly Predicted": cm.diagonal(),
        "Total in Test Set": cm.sum(axis=1),
        "Accuracy per Class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed Class Summary:")
    print(summary)


LSTM

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      0.32      0.39      1419
     neutral       0.54      0.76      0.63      4134
    positive       0.71      0.47      0.57      3570

    accuracy                           0.58      9123
   macro avg       0.58      0.52      0.53      9123
weighted avg       0.60      0.58      0.57      9123


Detailed Class Summary:
      Class  Correctly Predicted  Total in Test Set  Accuracy per Class
0  negative                  452               1419            0.318534
1   neutral                 3160               4134            0.764393
2  positive                 1683               3570            0.471429

Bi-LSTM

Classification Report:
              precision    recall  f1-score   support

    negative       0.51      0.33      0.40      1419
     neutral       0.63      0.67      0.65      4134
    positive       0.67      0.72      0.70      3570

    accuracy             

## 11. Visual analysis

## 12. Intepretability

In [94]:
text_vectorization_layer = lstm_model.layers[0] 
vocab = text_vectorization_layer.get_vocabulary()
print(f"Extracted vocabulary of size: {len(vocab)}")
index_to_word = {i: word for i, word in enumerate(vocab)}
class_names = ["Negative", "Neutral", "Positive"]

Extracted vocabulary of size: 10000


In [95]:
def analyze_and_log_lime_comprehensive(model, X_test_texts, run_name, class_names):
    print(f"[{run_name}] Generating LIME explanation")
    explainer = LimeTextExplainer(class_names=class_names)
    text_sample = X_test_texts[0]
    predict_fn = lambda x: model.predict(tf.constant(x, dtype=tf.string), verbose=0)
    
    exp = explainer.explain_instance(text_sample, predict_fn, num_features=10, top_labels=1)
    
    top_label_idx = exp.available_labels()[0]
    top_class_name = class_names[top_label_idx]
    
    html_name = f"lime_exp_{run_name}_class_{top_class_name}.html"
    exp.save_to_file(html_name)
    
    mlflow.log_artifact(html_name, artifact_path="interpretability/lime")
    if os.path.exists(html_name): os.remove(html_name)
    print(f"[{run_name}] LIME explanation logged.")

def analyze_and_log_shap_comprehensive(model, X_test_texts, run_name, class_names, index_to_word, num_samples_global=20):
    print(f"[{run_name}] Generating SHAP analysis")
    masker = shap.maskers.Text(r"\W+")
    
    def predict_fn(texts):
        if isinstance(texts, np.ndarray) and texts.dtype.kind == 'U':
            texts = texts.tolist()
        return model.predict(tf.constant(texts, dtype=tf.string), verbose=0)
    
    explainer = shap.Explainer(predict_fn, masker=masker, output_names=class_names)
    shap_dir = f"shap_artifacts_{run_name}"
    if not os.path.exists(shap_dir): os.makedirs(shap_dir)
        
    text_sample_local = X_test_texts[0]
    shap_values_local = explainer([text_sample_local])
    
    for i, class_name in enumerate(class_names):
        plt.figure(figsize=(10, 6))
        shap.plots.waterfall(shap_values_local[0, :, i], show=False)
        fig = plt.gcf()
        plt.title(f"SHAP Local: {class_name}", fontsize=12) 
        plt.tight_layout()
        plot_path = os.path.join(shap_dir, f"shap_waterfall_{class_name}.png")
        plt.savefig(plot_path)
        plt.close(fig)
        mlflow.log_artifact(plot_path, artifact_path="interpretability/shap/local_plots")

    test_subset_global = X_test_texts[:num_samples_global]
    shap_values_global = explainer(test_subset_global)
    
    word_shap_sums = {}
    for i in range(len(shap_values_global)):
        tokens, values = shap_values_global[i].data, shap_values_global[i].values
        for j, token in enumerate(tokens):
            clean_token = token.strip()
            if not clean_token: continue
            if clean_token not in word_shap_sums: word_shap_sums[clean_token] = np.zeros(len(class_names))
            word_shap_sums[clean_token] += np.abs(values[j])

    word_total_impact = {w: np.sum(word_shap_sums[w]) for w in word_shap_sums}
    top_words = sorted(word_total_impact, key=word_total_impact.get, reverse=True)[:15]
    top_words.reverse() 
    
    class_scores = {c: [] for c in class_names}
    for w in top_words:
        scores = word_shap_sums[w] / num_samples_global
        for k, c in enumerate(class_names):
            class_scores[c].append(scores[k])
            
    plt.figure(figsize=(10, 8))
    bottoms = np.zeros(len(top_words))
    for k, c in enumerate(class_names):
        plt.barh(top_words, class_scores[c], left=bottoms, label=c)
        bottoms += np.array(class_scores[c])
        
    plt.xlabel("Mean |SHAP value|")
    plt.title(f"Top 15 Global Features for {run_name}", fontsize=14)
    plt.legend()
    plt.tight_layout()
    plot_path_global = os.path.join(shap_dir, "shap_global_bar.png")
    plt.savefig(plot_path_global)
    plt.close()
    
    mlflow.log_artifact(plot_path_global, artifact_path="interpretability/shap")
    shutil.rmtree(shap_dir)
    print(f"[{run_name}] SHAP explanation logged.")

In [96]:
models_to_analyze = [
    (lstm_model, "LSTM"),
    (bilstm_model, "Bi-LSTM"),
    (ed_lstm_model, "ED-LSTM")
]

for model, run_name in models_to_analyze:
    print(f"\n{'='*60}")
    print(f"Starting analysis for model: {run_name}")
    print(f"{'='*60}")

    with mlflow.start_run(run_name=f"{run_name}_interpretability"):
        analyze_and_log_lime_comprehensive(model, X_test, run_name, class_names)
        analyze_and_log_shap_comprehensive(model, X_test, run_name, class_names, index_to_word, num_samples_global=20)

print("\nAll model interpretability analyses and artifacts logged to MLflow successfully.")


Starting analysis for model: LSTM
[LSTM] Generating LIME explanation
[LSTM] LIME explanation logged.
[LSTM] Generating SHAP analysis


PartitionExplainer explainer: 21it [00:21,  2.00s/it]                        


[LSTM] SHAP explanation logged.

Starting analysis for model: Bi-LSTM
[Bi-LSTM] Generating LIME explanation
[Bi-LSTM] LIME explanation logged.
[Bi-LSTM] Generating SHAP analysis


PartitionExplainer explainer: 21it [00:22,  2.01s/it]                        


[Bi-LSTM] SHAP explanation logged.

Starting analysis for model: ED-LSTM
[ED-LSTM] Generating LIME explanation
[ED-LSTM] LIME explanation logged.
[ED-LSTM] Generating SHAP analysis


PartitionExplainer explainer: 21it [00:22,  2.07s/it]                        

[ED-LSTM] SHAP explanation logged.

All model interpretability analyses and artifacts logged to MLflow successfully.
